# Topic Modeling

Discover themes and topics in social media posts using BERTopic results from the database
and sklearn LDA trained from scratch.

**What this notebook covers:**
- Explore pre-computed BERTopic topics from PostgreSQL
- Visualize topic sizes and keyword distributions
- Build a TF-IDF + LDA model from scratch on raw posts
- Compare BERTopic vs LDA topics
- Use LLM to generate human-readable topic names
- Word cloud of the dominant topic

In [ ]:
from workspace_utils import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from wordcloud import WordCloud

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. BERTopic Results from Database

In [ ]:
# Load pre-computed topics from PostgreSQL
topic_df = topics()
print(f'Found {len(topic_df)} topics in the database')
topic_df.head(10)

In [ ]:
# Bar chart of top 15 topics by size
top15 = topic_df.nlargest(15, 'size') if 'size' in topic_df.columns else topic_df.head(15)
size_col = 'size' if 'size' in topic_df.columns else topic_df.columns[-1]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    top15['name'].astype(str),
    top15[size_col],
    color=sns.color_palette('viridis', len(top15))
)
ax.set_xlabel('Number of Documents')
ax.set_title('Top 15 Topics by Size')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Show keywords per topic
kw_df = query_df("""
    SELECT t.topic_id, t.name, tk.keyword, tk.weight
    FROM topic_modeling.topics t
    JOIN topic_modeling.topic_keywords tk ON t.topic_id = tk.topic_id
    ORDER BY t.topic_id, tk.weight DESC
""")

# Display top keywords for each topic
for tid in kw_df['topic_id'].unique()[:10]:
    subset = kw_df[kw_df['topic_id'] == tid]
    name = subset['name'].iloc[0]
    words = ', '.join(subset['keyword'].head(8).tolist())
    print(f'Topic {tid} ({name}): {words}')

In [ ]:
# Plot topic network showing relationships between topics
plot_topic_network()

## 2. Build LDA from Scratch on Raw Posts

In [ ]:
# Load social posts for LDA training
posts = social_posts(limit=2000)
print(f'Loaded {len(posts)} posts for LDA training')

# Filter out empty content
texts = posts['content'].dropna().astype(str)
texts = texts[texts.str.len() > 20]
print(f'Using {len(texts)} non-empty posts')

In [ ]:
# TF-IDF vectorization
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    min_df=3,
    max_df=0.85,
    ngram_range=(1, 2)
)
tfidf_matrix = tfidf.fit_transform(texts)
feature_names = tfidf.get_feature_names_out()
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

# Fit LDA model
n_topics = 10
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=20,
    learning_method='online'
)
lda.fit(tfidf_matrix)
print(f'LDA model fitted with {n_topics} topics')
print(f'Log-likelihood: {lda.score(tfidf_matrix):.2f}')

In [ ]:
# Display LDA topics with top keywords
def display_lda_topics(model, feature_names, n_words=10):
    lda_topics = {}
    for idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        lda_topics[f'LDA Topic {idx}'] = top_words
        print(f'LDA Topic {idx}: {" | ".join(top_words)}')
    return lda_topics

lda_topics = display_lda_topics(lda, feature_names)

## 3. Compare BERTopic vs LDA

In [ ]:
# Compare BERTopic topics (from DB) with LDA topics (from scratch)
print('=== BERTopic Topics (from database) ===')
for _, row in topic_df.head(n_topics).iterrows():
    subset = kw_df[kw_df['topic_id'] == row['topic_id']]
    words = ', '.join(subset['keyword'].head(5).tolist())
    print(f"  {row['name']}: {words}")

print(f'\n=== LDA Topics (trained from scratch) ===')
for name, words in list(lda_topics.items())[:n_topics]:
    print(f'  {name}: {", ".join(words[:5])}')

print(f'\nBERTopic found {len(topic_df)} topics; LDA was configured for {n_topics}.')
print('BERTopic uses transformer embeddings; LDA uses bag-of-words TF-IDF.')

## 4. LLM-Assisted Topic Naming

In [ ]:
# Use the LLM to generate a human-readable name for the top LDA topic
top_topic_words = list(lda_topics.values())[0]
keywords = ', '.join(top_topic_words)

llm_name = ask_llm(
    f"Name this topic based on keywords: {keywords}. "
    f"Respond with just a short, descriptive 2-4 word topic name.",
    context=keywords
)
print(f'Top keywords: {keywords}')
print(f'LLM-suggested name: {llm_name}')

## 5. Word Cloud of Top Topic

In [ ]:
# Build word cloud from the largest BERTopic topic's documents
top_topic_id = topic_df.iloc[0]['topic_id'] if len(topic_df) > 0 else 0

topic_posts = query_df(f"""
    SELECT p.content
    FROM silver.posts p
    JOIN topic_modeling.document_topics dt ON p.post_id = dt.document_id
    WHERE dt.topic_id = {top_topic_id}
    LIMIT 500
""")

if len(topic_posts) > 0:
    all_text = ' '.join(topic_posts['content'].dropna().astype(str))
    wc = WordCloud(
        width=1000, height=500,
        background_color='white',
        colormap='viridis',
        max_words=80
    ).generate(all_text)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'Word Cloud - Topic: {topic_df.iloc[0]["name"]}', fontsize=16)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No documents found for the top topic. Falling back to all posts.')
    plot_wordcloud(posts['content'], title='Word Cloud - All Posts')

In [ ]:
# Summary
print('Topic Modeling Complete!')
print(f'  BERTopic topics in DB: {len(topic_df)}')
print(f'  LDA topics trained: {n_topics}')
print(f'  Posts analyzed: {len(texts)}')
print('\nNext: try text_classification.ipynb or cross_platform_analytics.ipynb')